In [ ]:
## Harmony integration
## imports module
import os
import scanpy as sc
import scvi
import seaborn as sns
import pandas as pd
from rich import print
import numpy as np
import psutil, gc
import  anndata as ad

In [ ]:
# check how many memory used
print(psutil.virtual_memory())
import torch
# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available")
    print(torch.version.cuda)
    # Get the number of available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")
    
    # Get the name of each available GPU
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")
else:
    print("CUDA is not available")

In [ ]:
## general setting
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/scratch/users/aliu10/vascular"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme(palette='viridis')
torch.set_float32_matmul_precision("high")

In [ ]:
# --- load query data ---
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_03032026.h5ad")
print(adata)
print(adata.X[:20,:20])

## Plotting the overall project with cell type labeling

In [ ]:
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures/"
sc.set_figure_params(figsize=(10, 10), frameon=False)
sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color=['Cell_type'],
    frameon=False,
    ncols=4,
    size=5,
    legend_loc="on data",
    legend_fontweight='light',
    save=f"_overall_data_cell_type.svg" 
)


## Loadig the published data

In [ ]:
## Load data from other datasets
adata_1 = sc.read_h5ad("./Data/Walchli_data/Walchli_data_sorted_EC.h5ad")
adata_2 = sc.read_h5ad("./Data/Walchli_data/Walchli_data_unsorted.h5ad")
adata_3 = sc.read_h5ad("./Data/AY_data/AY_data.h5ad")
adata_4 = sc.read_h5ad("./Data/Garcia_data/Garcia_data.h5ad")
adata_5 = sc.read_h5ad("./Data/Sun_data/Sun_data.h5ad")

# print(adata)

## check if count data
print(adata_1.X[:5,:5])
print(adata_2.X[:5,:5])
print(adata_3.X[:5,:10])
print(adata_4.X[:5,:5])
print(adata_5.X[:5,:5])

In [ ]:
## Check the var names
var_IH = adata.var_names.to_list()
print(len(var_IH))
var_1 = adata_1.var_names.to_list()
print(len(var_1))
var_2 = adata_2.var_names.to_list()
print(len(var_2))
var_3 = adata_3.var_names.to_list()
print(len(var_3))
var_4 = adata_4.var_names.to_list()
print(len(var_4))
var_5 = adata_5.var_names.to_list()
print(len(var_5))

overlap = list(set(var_IH) & set(var_1) & set(var_2) & set(var_3) & set(var_4) & set(var_5))
print(len(overlap))



In [ ]:
### Checking the cell types of each object
print(adata_1.obs['Cell_type'].value_counts())
print(adata_2.obs['Cell_type'].value_counts())
print(adata_3.obs['Cell_type'].value_counts())
print(adata_4.obs['Cell_type'].value_counts())
print(adata_5.obs['Cell_type'].value_counts())

# print(adata_1.obs['Cell_class'].value_counts())
print(adata_2.obs['Cell_class'].value_counts())
print(adata_3.obs['Cell_class'].value_counts())
print(adata_4.obs['Cell_class'].value_counts())
print(adata_5.obs['Cell_class'].value_counts())

## Organizing all the vascular file

In [ ]:
## IH data first
adata.obs['Cell_type'] = adata.obs['cell.class_after_decontX'].copy()
adata.obs['Cell_class'] = adata.obs['cell.class_after_decontX'].copy()
## check
print(adata.obs['Cell_class'].value_counts())

In [ ]:
meta = adata.obs[['individualID',"ageatdeath","sex","brain_region",'nCount_RNA','nFeature_RNA',"Cell_type","Cell_class"]]
meta["Study"] = "IH"

meta.rename(columns={'ageatdeath': 'Age','sex':'Sex','brain_region':'Brain_region'}, inplace=True)
adata.obs = meta

## subsetting
adata = adata[adata.obs["Cell_class"].isin(["Endothelial","Mural_Cell","Fibroblast"])].copy()
print(adata)
print(adata.obs.head())

In [ ]:
# ## working on IH_flex
# meta = adata_6.obs[['individualID',"AgeDeath","Sex","region_abb",'nCount_RNA','nFeature_RNA','Cell_type']]
# meta["Cell_class"] = "Endothelial"
# meta["Study"] = "IH_FLEX"

# meta.rename(columns={'AgeDeath': 'Age','region_abb':'Brain_region'}, inplace=True)

# adata_6.obs = meta
# adata_6.obs.head()

In [ ]:
adata_2 = adata_2[adata_2.obs["Cell_class"].isin(["Endothelial","Mural_Cell","Fibroblast"])].copy()
adata_3 = adata_3[adata_3.obs["Cell_class"].isin(["Endothelial","Mural_Cell","Fibroblast"])].copy()
adata_4 = adata_4[adata_4.obs["Cell_class"].isin(["Endothelial","Mural_Cell","Fibroblast"])].copy()
adata_5 = adata_5[adata_5.obs["Cell_class"].isin(["Endothelial","Mural_Cell","Fibroblast"])].copy()


adata_1.obs["Study"] = "Walchli_sorted"
adata_1.obs["Cell_class"] = "Endothelial"
adata_2.obs["Study"] = "Walchli_unsorted"
adata_3.obs["Study"] = "Yang"
adata_4.obs["Study"] = "Garcia"
adata_5.obs["Study"] = "Sun"

print(adata_1)
print(adata_2)
print(adata_3)
print(adata_4)
print(adata_5)

## Concate all data together

In [ ]:

# # adata = ad.concat([adata,adata_1,adata_2,adata_3,adata_4,adata_5,adata_6],join = "outer")
adata = ad.concat([adata,adata_1,adata_2,adata_3,adata_4,adata_5],join = "inner")
print(adata)

###before subsetting
print("Before age subsetting")
print(adata.obs["Study"].value_counts())
adata.obs['Age'] = pd.to_numeric(adata.obs['Age'], errors='coerce')
meta = adata.obs.copy()
print(pd.crosstab(meta['Study'],meta['Cell_class']))

## Try to see how many left after age subsetting
meta = adata.obs.copy()
meta = meta[(meta["Age"] >= 30) & (meta["Age"] <= 65)].copy()
print("After age subsetting")
print(meta['Study'].value_counts())
print(pd.crosstab(meta['Study'],meta['Cell_class']))

In [ ]:
## save again
adata.write_h5ad("./Results/Revision_2/All_datasets_vascular_raw.h5ad")

## Preprocessing files for integration

In [ ]:
## reloading if needed
adata = sc.read_h5ad("./Data/All_datasets_vascular_raw.h5ad")
adata.obs_names_make_unique()
print(adata)

In [ ]:
## exclude nuclei belong to IH but not in meta I had before
meta_new = adata.obs.copy()
meta_new = meta_new[meta_new["Study"] == "IH"].copy()
meta_new = meta_new[~meta_new.index.isin(meta.index)].copy()
meta_new.shape

## subset adata by keeping only the nuclei not in meta_new
adata = adata[~adata.obs.index.isin(meta_new.index)].copy()
print(adata)

In [ ]:
### subsetting for endothelial
adata = adata[adata.obs["Cell_class"] == "Endothelial"].copy()
print(adata)

## Filter by age

In [ ]:
## Filter by age
adata = adata[(adata.obs["Age"] >= 30) & (adata.obs["Age"] <= 65), :].copy()
print(adata)
print(adata.X[:10,:10])

In [ ]:
# Count nuclei per individual
counts = adata.obs["individualID"].value_counts()

# Keep individuals with more than one nucleus
valid_individuals = counts[counts > 1].index

# Subset AnnData
adata = adata[adata.obs["individualID"].isin(valid_individuals)].copy()

# Check data
print(adata)

In [ ]:
adata.var_names_make_unique()
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

print(adata)

In [ ]:
## save backup
adata.raw = adata.copy()

## get the counts layer
adata.layers["counts"] = adata.X.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 
print(adata)

In [ ]:
adata.obs['individualID'].value_counts()
pd.crosstab(adata.obs['Study'], adata.obs['individualID'])

In [ ]:
## Redo the normalization
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)

## only using the highly variable genes for integration task
### using the top 3000 genes for scVI training
adata_scvi = adata[:,adata.var["highly_variable"]].copy()
print(adata_scvi)

In [ ]:
### ---------------------------------------------------------------------
### Second, scVI model training
### ---------------------------------------------------------------------
## uing counts for integration tasks
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key="individualID",
    categorical_covariate_keys=['Study','individualID'],
    continuous_covariate_keys=['total_counts','pct_counts_mt']
    )
scvi_model = scvi.model.SCVI(adata_scvi, n_layers=2, n_latent=30, gene_likelihood="nb")
# scvi_model.view_anndata_setup()

In [ ]:
## train scvi model first for integration
scvi_model.train() #batch_size=125

## assign the scvi latent key to the anndata 
SCVI_LATENT_KEY = "X_scVI"
adata_scvi.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()

In [ ]:
# find neighbors
sc.pp.neighbors(adata_scvi, use_rep=SCVI_LATENT_KEY)
# get clusters
# sc.tl.leiden(adata_scvi,resolution=0.8,key_added='leiden_scVI')

print(adata_scvi)

In [ ]:
# where to save figures
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures_all_datasets"
os.makedirs(sc.settings.figdir, exist_ok=True)

print(f"Computing UMAP")
key = "umap_scvi"
sc.tl.umap(
    adata_scvi,
    random_state=42,
    key_added=key
)

# Clear any cached color palettes for the labels you plot
for k in ["individualID", "Study","Cell_type"]:
    adata_scvi.uns.pop(f"{k}_colors", None)

# Use the generic embedding plotter to select your custom basis
sc.pl.embedding(
    adata_scvi,
    basis=key,
    color=["individualID", "Study","Cell_class",'Cell_type'],
    frameon=False,
    ncols=4,
    size=5,
    legend_loc="on data",
    # save=f"_age_integrated_Endo.svg" 
)
# print(f"Saved UMAP -> embedding_{key}_Endo.svg")


In [ ]:
### ---------------------------------------------------------------------
### Forth, get the object and save
### ---------------------------------------------------------------------
adata = adata_scvi.raw.to_adata()
print(adata)
## Saving h5ad
results_file = PATH+"/Results/Revision_2/All_datasets_vascular_age_integrated.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Harmony integration

In [ ]:
print(adata)
print(adata.X[:20,:20])

In [ ]:
# ## keeping backup
adata.raw = adata.copy()

# ## normalization
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

In [ ]:
# ## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=1500,
    subset=True,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)
print(adata)

In [ ]:
# ### pca and integration
print("Running PCA")
sc.pp.pca(adata)
rep = "X_pca"
# print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key=['individualID'], max_iter_harmony=50)
rep = "X_pca_harmony"

In [ ]:
# print("Computing neighbors")
sc.pp.neighbors(adata, use_rep=rep)
print("Computing leiden")
sc.tl.leiden(adata,resolution=0.8,key_added='leiden_harmony')

In [ ]:
# where to save figures
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures_all_datasets"
os.makedirs(sc.settings.figdir, exist_ok=True)

print(f"Computing UMAP")
sc.tl.umap(
    adata, min_dist=0.8, spread=1.0, 
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

# Clear any cached color palettes for the labels you plot
for k in ["individualID", "Study","Cell_type"]:
    adata.uns.pop(f"{k}_colors", None)

# Use the generic embedding plotter to select your custom basis
sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color=["individualID", "Study","Cell_class",'Cell_type'],
    frameon=False,
    ncols=3,
    size=2,
    legend_loc="on data",
    save=f"_age_integrated_Endo.svg" 
)
print(f"Saved UMAP -> embedding_age_integrated_Endo.svg")


In [ ]:
### ---------------------------------------------------------------------
### Forth, get the object and save
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()
print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/All_datasets_vascular_age_integrated_Endo.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Reloading the data set and plotting

In [ ]:
# adata = sc.read_h5ad(PATH+"/Results/Revision_2/All_datasets_vascular_age_integrated_Endo.h5ad")
print(adata)
print(adata.X[:20,:20])

In [ ]:
adata.obs["Sex"] = adata.obs["Sex"].replace({"Male": "M"})
adata.obs["Sex"] = adata.obs["Sex"].replace({"Female": "F"})

In [ ]:
adata.obs["Sex"].value_counts()

In [ ]:
## check how many cells in each dataset left
print(adata.obs["Study"].value_counts())

In [ ]:
adata = adata.raw.to_adata()
print(adata)
print(adata.X[:20,:20])

## Reloading for plotting

In [ ]:
### Reloading the data
adata = sc.read_h5ad(PATH+"/Results/Revision_2/All_datasets_vascular_age_integrated_Endo.h5ad")
print(adata)
print(adata.X[:20,:20]) 

In [ ]:
# Study colors
col1 = {
    "IH"                : "#44355b",
    "Walchli_sorted"    : "#e95500",
    "Walchli_unsorted"  : "#ee5622",
    "Yang"              : "#eca72c"
}

# Cell type colors
col2 = {
    "Large artery"          : "#B2182B",
    "Large_Artery"          : "#B2182B",
    "Artery"                : "#D6604D",
    "Arterial"              : "#D6604D",
    "Arteriole"             : "#F4A582",
    "Capillary"             : "#A6DBD0",
    "Endothelial cells"     : "#A6DBD0",
    "Angiogenic capillary"  : "#66C2A5",
    "Capillary3"            : "#41AE76",
    "Venule"                : "#4393C3",
    "Venous"                : "#4393C3",
    "Large vein"            : "#2166AC",
    "Vein"                  : "#2166AC",
    "Fenestrated_Capillary" : "#FDAE61",
    "EndoMT"                : "#BC80BD",
    "Proliferating cell"    : "#762A83",
    "Mitochondrial"         : "#574b5a",
}

In [ ]:
# Define study of interest
study_oi = "Yang"

# Create new column copying Cell_type
adata.obs['Cell_type_study'] = adata.obs['Cell_type'].astype(str)

# Set all cells not from study of interest to "Other"
adata.obs.loc[adata.obs['Study'] != study_oi, 'Cell_type_study'] = np.nan

# Convert to categorical with "Other" (nan) plotted beneath
adata.obs['Cell_type_study'] = pd.Categorical(
    adata.obs['Cell_type_study'],
    categories=[c for c in adata.obs['Cell_type'].cat.categories]  # original cell type order
)

# Plot — na_color handles "Other" as grey beneath all cell types automatically
sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color="Cell_type_study",
    na_color="lightgrey",         
    # palette=col2,              # your color dictionary
    frameon=False,
    # size=10,
    legend_loc="on data",
    rasterized=True,
    # save=f"_umap_{study_oi}.svg"
)

In [ ]:
exit()